# 🎲 Distribuições e Períodos em Geradores Pseudoaleatórios

> Dois conceitos fundamentais para entender como `random.Random` funciona de verdade.

---

## Parte 1 — Distribuição
## Parte 2 — Período

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

---
# PARTE 1 — Distribuição

## Conceito

O Mersenne Twister gera números em sua escala interna (~32 bits).
O `randint(a, b)` **não muda a sequência interna** — ele apenas mapeia esses números para o intervalo `[a, b]`.

```
Sequência interna:    [3891650795,  1908627132,  2756349801, ...]
                             ↓             ↓              ↓
randint(0, 100):            49,           97,            53
randint(101, 200):         150,          198,           154
```

**Consequência:** mudar o intervalo muda os *valores que você vê*, mas não a *fila interna*.
Sequências com a mesma semente e intervalos diferentes são **transformações do mesmo processo**.

## 1.1 — O intervalo muda os valores, mas preserva a forma da distribuição

In [1]:
N = 10_000
intervalos = [(0, 100), (101, 200), (0, 1000), (500, 600)]

fig, axes = plt.subplots(2, 2, figsize=(13, 7))
fig.suptitle("randint(a, b) com mesma semente — a distribuição é sempre Uniforme(a, b)",
             fontweight='bold')

for ax, (a, b) in zip(axes.flat, intervalos):
    rng = random.Random(42)
    amostra = [rng.randint(a, b) for _ in range(N)]

    ax.hist(amostra, bins=40, color='#2196F3', alpha=0.8, edgecolor='white', density=True)

    # Linha da distribuição uniforme teórica
    altura_teorica = 1 / (b - a + 1)
    ax.axhline(altura_teorica, color='#E53935', linewidth=2,
               linestyle='--', label=f'Uniforme teórica\n(altura={altura_teorica:.5f})')

    ax.set_title(f'randint({a}, {b})')
    ax.set_xlabel('Valor')
    ax.set_ylabel('Densidade')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("Observação: independente do intervalo, o histograma sempre se aproxima")
print("da distribuição Uniforme — a forma não muda, apenas a escala.")

NameError: name 'plt' is not defined

## 1.2 — A correlação entre intervalos diferentes com a mesma semente

Como os valores de dois intervalos diferentes (mesma semente) vêm da **mesma fila interna**,
eles são correlacionados — um sobe quando o outro sobe.

In [ ]:
N = 500

# Mesma semente, intervalos diferentes
rng_a = random.Random(0)
seq_0_100 = [rng_a.randint(0, 100) for _ in range(N)]

rng_b = random.Random(0)  # reinicia a mesma fila
seq_101_200 = [rng_b.randint(101, 200) for _ in range(N)]

# Semente diferente para comparação
rng_c = random.Random(99)
seq_independente = [rng_c.randint(0, 100) for _ in range(N)]

corr_mesma, _  = stats.pearsonr(seq_0_100, seq_101_200)
corr_dif, _    = stats.pearsonr(seq_0_100, seq_independente)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Correlação entre sequências: mesma semente vs. semente diferente",
             fontweight='bold')

# Plot 1: mesma semente
axes[0].scatter(seq_0_100, seq_101_200, alpha=0.3, s=15, color='#E53935')
axes[0].set_title(f'Mesma semente (r = {corr_mesma:.3f})')
axes[0].set_xlabel('randint(0, 100)')
axes[0].set_ylabel('randint(101, 200)')

# Linha de tendência
m, c = np.polyfit(seq_0_100, seq_101_200, 1)
xs = np.linspace(0, 100, 100)
axes[0].plot(xs, m * xs + c, color='black', linewidth=2, label=f'tendência')
axes[0].legend()

# Plot 2: semente diferente
axes[1].scatter(seq_0_100, seq_independente, alpha=0.3, s=15, color='#2196F3')
axes[1].set_title(f'Semente diferente (r = {corr_dif:.3f})')
axes[1].set_xlabel('randint(0, 100) — semente 0')
axes[1].set_ylabel('randint(0, 100) — semente 99')

m2, c2 = np.polyfit(seq_0_100, seq_independente, 1)
axes[1].plot(xs, m2 * xs + c2, color='black', linewidth=2, label='tendência')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f"Correlação (mesma semente, intervalos diferentes): r = {corr_mesma:.4f}")
print(f"Correlação (sementes diferentes):                  r = {corr_dif:.4f}")
print()
print("→ A correlação próxima de 1 na esquerda revela que os dois intervalos")
print("  são transformações lineares da mesma sequência interna.")

---
# PARTE 2 — Período

## Conceito

O **período** de um gerador é o número de elementos que ele produz antes de **repetir a sequência do início**.

O Mersenne Twister tem período $2^{19937} - 1$ — na prática, infinito.
Mas geradores mais simples têm períodos pequenos e **demonstráveis**.

Vamos usar um **Gerador Congruencial Linear (LCG)** — o algoritmo mais simples possível — para tornar o período visível.

## 2.1 — Implementando um LCG simples

A fórmula do LCG é:

$$X_{n+1} = (a \cdot X_n + c) \mod m$$

Onde:
- $m$ = módulo (tamanho do espaço de estados)
- $a$ = multiplicador
- $c$ = incremento
- $X_0$ = semente

In [ ]:
def lcg(semente, a, c, m, n):
    """Gera n números com um LCG e retorna a sequência."""
    sequencia = []
    x = semente
    for _ in range(n):
        x = (a * x + c) % m
        sequencia.append(x)
    return sequencia

def encontrar_periodo(semente, a, c, m):
    """Encontra o período real do gerador."""
    x = semente
    visitados = {}
    passo = 0
    while x not in visitados:
        visitados[x] = passo
        x = (a * x + c) % m
        passo += 1
    return passo - visitados[x], visitados[x]  # período, ponto de entrada

# Exemplo com período curto — fácil de visualizar
semente, a, c, m = 1, 5, 3, 16
seq = lcg(semente, a, c, m, n=30)
periodo, entrada = encontrar_periodo(semente, a, c, m)

print(f"Parâmetros: a={a}, c={c}, m={m}, semente={semente}")
print(f"Sequência gerada (30 valores):")
print(seq)
print()
print(f"→ Período encontrado: {periodo} (a sequência se repete a cada {periodo} números)")
print(f"→ A repetição começa na posição: {entrada}")

## 2.2 — Visualizando o período

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"LCG com a={a}, c={c}, m={m} — Período = {periodo}", fontweight='bold')

cores = ['#2196F3', '#E53935', '#4CAF50']
n_ciclos = 3

# Gráfico esquerdo: série temporal colorida por ciclo
seq_longa = lcg(semente, a, c, m, n=periodo * n_ciclos)
for i in range(n_ciclos):
    trecho = seq_longa[i*periodo:(i+1)*periodo]
    xs = range(i*periodo, (i+1)*periodo)
    axes[0].plot(list(xs), trecho, color=cores[i], marker='o',
                 markersize=5, linewidth=1.5, label=f'Ciclo {i+1}')
    # Marca a fronteira do período
    if i > 0:
        axes[0].axvline(x=i*periodo, color='black', linestyle='--', linewidth=1)

axes[0].set_title("Série temporal — os ciclos se repetem identicamente")
axes[0].set_xlabel("Posição na sequência")
axes[0].set_ylabel("Valor")
axes[0].legend()

# Gráfico direito: espaço de estados (X_n vs X_{n+1})
seq_um_ciclo = lcg(semente, a, c, m, n=periodo+1)
axes[1].scatter(seq_um_ciclo[:-1], seq_um_ciclo[1:],
                color='#9C27B0', s=80, zorder=3)
for i in range(len(seq_um_ciclo)-1):
    axes[1].annotate('', xy=(seq_um_ciclo[i+1], seq_um_ciclo[i+2] if i+2 < len(seq_um_ciclo) else seq_um_ciclo[1]),
                     xytext=(seq_um_ciclo[i], seq_um_ciclo[i+1]),
                     arrowprops=dict(arrowstyle='->', color='#9C27B0', lw=1.2))

axes[1].set_title(f"Espaço de estados: X_n vs X_{{n+1}} — ciclo fechado com {periodo} estados")
axes[1].set_xlabel("X_n")
axes[1].set_ylabel("X_{n+1}")

plt.tight_layout()
plt.show()

## 2.3 — Como os parâmetros afetam o período

Pequenas mudanças em `a`, `c` ou `m` podem **reduzir drasticamente** o período — tornando o gerador péssimo.

In [ ]:
configuracoes = [
    (5,  3, 16, "Bom — período máximo"),
    (2,  3, 16, "Ruim — período curto"),
    (4,  1, 16, "Péssimo — colapsa rápido"),
    (5,  3, 32, "Maior m — período maior"),
]

print(f"{'Parâmetros (a, c, m)':<25} {'Período':>10}  Descrição")
print("-" * 65)
for a_, c_, m_, desc in configuracoes:
    p, _ = encontrar_periodo(1, a_, c_, m_)
    print(f"a={a_}, c={c_}, m={m_:<6}          {p:>6}    {desc}")

print()
print("Regra geral: o período máximo possível de um LCG é m.")
print("Para atingir período máximo, a, c e m precisam satisfazer o Teorema de Hull-Dobell.")

## 2.4 — Por que o Mersenne Twister nunca repete na prática

In [ ]:
import math

periodo_mt = 2**19937 - 1
atomos_universo = 10**80
nanossegundos_universo = 13.8e9 * 365.25 * 24 * 3600 * 1e9  # ~4.4e26 ns

print("Período do Mersenne Twister: 2^19937 - 1")
print(f"Número de dígitos: ~{len(str(periodo_mt))}")
print()
print(f"Átomos no universo observável:      ~10^80")
print(f"Nanossegundos desde o Big Bang:     ~4.4 x 10^26")
print(f"Dígitos do período do MT:           ~{len(str(periodo_mt))} → 10^6001")
print()
print("Se você gerasse 1 número por nanossegundo desde o Big Bang:")
numeros_gerados = nanossegundos_universo
print(f"  Números gerados até hoje: ~{numeros_gerados:.2e}")
print(f"  Período do MT:            ~10^6001")
print(f"  Fração percorrida:        ~10^{math.log10(numeros_gerados) - 6001:.0f}")
print()
print("→ Na prática, o período do Mersenne Twister é efetivamente infinito.")

---
## 📝 Resumo Final

| Conceito | O que é | Como verificar |
|---|---|---|
| **Distribuição** | A forma como os valores se espalham no intervalo | Histograma — sempre Uniforme(a,b) |
| **Correlação entre intervalos** | Mesma semente → mesma fila interna → valores correlacionados | Scatter plot + Pearson |
| **Período** | Quantos números até a sequência se repetir | Detectável em LCGs simples |
| **Período do MT** | $2^{19937}-1$ | Efetivamente infinito para qualquer uso real |

> **Regra de ouro:** mudar o intervalo `[a, b]` muda a escala, não o gerador.
> Mudar a **semente** é o único jeito de obter sequências verdadeiramente independentes.